# Retail Sales Data Analysis
## Comprehensive Exploratory Data Analysis
**Author:** Aman Panday  
**Project:** Retail Sales Analytics Dashboard  
**Date:** March 2026

---

## Table of Contents
1. [Import Libraries](#import)
2. [Load Data](#load)
3. [Data Overview](#overview)
4. [Revenue Analysis](#revenue)
5. [Regional Performance](#regional)
6. [Product Category Analysis](#product)
7. [Customer Segmentation](#customer)
8. [Time Series Analysis](#timeseries)
9. [Key Insights](#insights)

## 1. Import Required Libraries <a id='import'></a>

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully!")

## 2. Load Cleaned Data <a id='load'></a>

In [ ]:
# Load the cleaned sales data
df = pd.read_csv('../data/cleaned_sales_data.csv')

# Convert Order Date to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'])

print(f"Dataset Shape: {df.shape}")
print(f"Total Records: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")
print("\nFirst few records:")
df.head()

## 3. Data Overview <a id='overview'></a>

In [ ]:
# Display basic information
print("Data Types:")
print(df.dtypes)
print("\n" + "="*60)
print("Missing Values:")
print(df.isnull().sum())
print("\n" + "="*60)
print("Unique Values:")
print(df.nunique())

In [ ]:
# Summary statistics
print("Summary Statistics:")
df.describe()

## 4. Revenue Analysis <a id='revenue'></a>

In [ ]:
# Calculate key metrics
total_revenue = df['Sales'].sum()
total_profit = df['Profit'].sum()
total_orders = df['Order ID'].nunique()
avg_order_value = df['Sales'].mean()
profit_margin = (total_profit / total_revenue) * 100

print("🔑 KEY PERFORMANCE INDICATORS")
print("="*60)
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Average Order Value: ${avg_order_value:,.2f}")
print(f"Overall Profit Margin: {profit_margin:.2f}%")

In [ ]:
# Monthly revenue trend
monthly_revenue = df.groupby('Month Name')['Sales'].sum().reindex([
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
])

plt.figure(figsize=(14, 6))
plt.plot(monthly_revenue.index, monthly_revenue.values, marker='o', linewidth=2, markersize=8, color='#2ecc71')
plt.title('Monthly Revenue Trend', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Month', fontsize=12)
plt.ylabel('Revenue ($)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Add value labels on points
for i, v in enumerate(monthly_revenue.values):
    plt.text(i, v, f'${v:,.0f}', ha='center', va='bottom', fontsize=9)

plt.show()

print("\nMonthly Revenue:")
print(monthly_revenue)

In [ ]:
# Quarterly performance
quarterly_data = df.groupby('Quarter').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order ID': 'nunique'
}).round(2)

quarterly_data.columns = ['Revenue', 'Profit', 'Orders']
quarterly_data['Profit Margin %'] = ((quarterly_data['Profit'] / quarterly_data['Revenue']) * 100).round(2)

print("Quarterly Performance:")
print(quarterly_data)

# Visualize quarterly performance
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Revenue and Profit by Quarter
quarterly_data[['Revenue', 'Profit']].plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c'])
axes[0].set_title('Quarterly Revenue and Profit', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Quarter', fontsize=11)
axes[0].set_ylabel('Amount ($)', fontsize=11)
axes[0].legend(['Revenue', 'Profit'])
axes[0].tick_params(axis='x', rotation=0)

# Profit Margin by Quarter
quarterly_data['Profit Margin %'].plot(kind='line', marker='o', ax=axes[1], color='#9b59b6', linewidth=2, markersize=10)
axes[1].set_title('Quarterly Profit Margin Trend', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Quarter', fontsize=11)
axes[1].set_ylabel('Profit Margin (%)', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Regional Performance <a id='regional'></a>

In [ ]:
# Regional analysis
regional_performance = df.groupby('Region').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order ID': 'nunique',
    'Quantity': 'sum'
}).round(2)

regional_performance.columns = ['Revenue', 'Profit', 'Orders', 'Units Sold']
regional_performance['Avg Order Value'] = (regional_performance['Revenue'] / regional_performance['Orders']).round(2)
regional_performance = regional_performance.sort_values('Revenue', ascending=False)

print("Regional Performance Analysis:")
print(regional_performance)

# Calculate percentage contribution
regional_performance['Revenue %'] = ((regional_performance['Revenue'] / regional_performance['Revenue'].sum()) * 100).round(2)
print("\nRevenue Contribution by Region:")
print(regional_performance[['Revenue', 'Revenue %']])

In [ ]:
# Visualize regional performance
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Revenue by Region - Bar Chart
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
regional_performance['Revenue'].plot(kind='bar', ax=axes[0, 0], color=colors)
axes[0, 0].set_title('Revenue by Region', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Region', fontsize=11)
axes[0, 0].set_ylabel('Revenue ($)', fontsize=11)
axes[0, 0].tick_params(axis='x', rotation=0)

# Revenue Distribution - Pie Chart
axes[0, 1].pie(regional_performance['Revenue'], labels=regional_performance.index, autopct='%1.1f%%',
               colors=colors, startangle=90, textprops={'fontsize': 11})
axes[0, 1].set_title('Revenue Distribution by Region', fontsize=14, fontweight='bold')

# Orders by Region
regional_performance['Orders'].plot(kind='barh', ax=axes[1, 0], color=colors)
axes[1, 0].set_title('Number of Orders by Region', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Number of Orders', fontsize=11)
axes[1, 0].set_ylabel('Region', fontsize=11)

# Profit by Region
regional_performance['Profit'].plot(kind='bar', ax=axes[1, 1], color=colors, alpha=0.7)
axes[1, 1].set_title('Profit by Region', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Region', fontsize=11)
axes[1, 1].set_ylabel('Profit ($)', fontsize=11)
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 6. Product Category Analysis <a id='product'></a>

In [ ]:
# Product category performance
category_performance = df.groupby('Product Category').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Quantity': 'sum',
    'Order ID': 'nunique'
}).round(2)

category_performance.columns = ['Revenue', 'Profit', 'Units Sold', 'Orders']
category_performance['Profit Margin %'] = ((category_performance['Profit'] / category_performance['Revenue']) * 100).round(2)
category_performance = category_performance.sort_values('Revenue', ascending=False)

print("Product Category Performance:")
print(category_performance)

In [ ]:
# Visualize product category performance
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Revenue by Category
category_performance['Revenue'].plot(kind='bar', ax=axes[0, 0], color=['#9b59b6', '#1abc9c', '#e67e22', '#34495e'])
axes[0, 0].set_title('Revenue by Product Category', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Category', fontsize=11)
axes[0, 0].set_ylabel('Revenue ($)', fontsize=11)
axes[0, 0].tick_params(axis='x', rotation=45)

# Profit Margin Comparison
category_performance['Profit Margin %'].plot(kind='bar', ax=axes[0, 1], color='#e74c3c', alpha=0.7)
axes[0, 1].set_title('Profit Margin by Category', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Category', fontsize=11)
axes[0, 1].set_ylabel('Profit Margin (%)', fontsize=11)
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].axhline(y=category_performance['Profit Margin %'].mean(), color='red', linestyle='--', label='Average')
axes[0, 1].legend()

# Units Sold by Category
category_performance['Units Sold'].plot(kind='barh', ax=axes[1, 0], color='#3498db')
axes[1, 0].set_title('Units Sold by Category', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Units Sold', fontsize=11)
axes[1, 0].set_ylabel('Category', fontsize=11)

# Category Revenue Distribution
axes[1, 1].pie(category_performance['Revenue'], labels=category_performance.index, autopct='%1.1f%%',
               colors=['#9b59b6', '#1abc9c', '#e67e22', '#34495e'], startangle=90, textprops={'fontsize': 10})
axes[1, 1].set_title('Revenue Share by Category', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 products by revenue
top_products = df.groupby('Product Name')['Sales'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
top_products.plot(kind='barh', color='steelblue')
plt.title('Top 10 Products by Revenue', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Revenue ($)', fontsize=12)
plt.ylabel('Product Name', fontsize=12)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 Products by Revenue:")
print(top_products)

## 7. Customer Segmentation Analysis <a id='customer'></a>

In [ ]:
# Customer segment performance
segment_performance = df.groupby('Customer Segment').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order ID': 'nunique',
    'Quantity': 'sum'
}).round(2)

segment_performance.columns = ['Revenue', 'Profit', 'Orders', 'Units Sold']
segment_performance['Avg Order Value'] = (segment_performance['Revenue'] / segment_performance['Orders']).round(2)
segment_performance['Profit Margin %'] = ((segment_performance['Profit'] / segment_performance['Revenue']) * 100).round(2)
segment_performance = segment_performance.sort_values('Revenue', ascending=False)

print("Customer Segment Performance:")
print(segment_performance)

In [ ]:
# Visualize customer segment analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Revenue by Customer Segment
segment_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
segment_performance['Revenue'].plot(kind='bar', ax=axes[0, 0], color=segment_colors)
axes[0, 0].set_title('Revenue by Customer Segment', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Customer Segment', fontsize=11)
axes[0, 0].set_ylabel('Revenue ($)', fontsize=11)
axes[0, 0].tick_params(axis='x', rotation=45)

# Average Order Value by Segment
segment_performance['Avg Order Value'].plot(kind='bar', ax=axes[0, 1], color='#95E1D3')
axes[0, 1].set_title('Average Order Value by Segment', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Customer Segment', fontsize=11)
axes[0, 1].set_ylabel('Avg Order Value ($)', fontsize=11)
axes[0, 1].tick_params(axis='x', rotation=45)

# Segment Distribution - Pie Chart
axes[1, 0].pie(segment_performance['Revenue'], labels=segment_performance.index, autopct='%1.1f%%',
               colors=segment_colors, startangle=90, textprops={'fontsize': 11})
axes[1, 0].set_title('Revenue Distribution by Segment', fontsize=14, fontweight='bold')

# Profit Margin by Segment
segment_performance['Profit Margin %'].plot(kind='bar', ax=axes[1, 1], color='#F38181', alpha=0.7)
axes[1, 1].set_title('Profit Margin by Customer Segment', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Customer Segment', fontsize=11)
axes[1, 1].set_ylabel('Profit Margin (%)', fontsize=11)
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 8. Time Series Analysis <a id='timeseries'></a>

In [ ]:
# Daily sales trend
daily_sales = df.groupby('Order Date').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order ID': 'nunique'
}).reset_index()

plt.figure(figsize=(16, 6))
plt.plot(daily_sales['Order Date'], daily_sales['Sales'], linewidth=1.5, color='#3498db', alpha=0.8)
plt.title('Daily Sales Trend', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Sales ($)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Day of week analysis
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_sales = df.groupby('Day of Week')['Sales'].sum().reindex(day_order)

plt.figure(figsize=(12, 6))
colors_dow = ['#FF6B6B' if x == dow_sales.max() else '#95E1D3' for x in dow_sales]
dow_sales.plot(kind='bar', color=colors_dow)
plt.title('Sales by Day of Week', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Day of Week', fontsize=12)
plt.ylabel('Total Sales ($)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nSales by Day of Week:")
print(dow_sales)

In [ ]:
# Heatmap: Sales by Region and Category
heatmap_data = df.pivot_table(values='Sales', index='Region', columns='Product Category', aggfunc='sum', fill_value=0)

plt.figure(figsize=(12, 6))
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5, cbar_kws={'label': 'Revenue ($)'})
plt.title('Sales Heatmap: Region vs Product Category', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Product Category', fontsize=12)
plt.ylabel('Region', fontsize=12)
plt.tight_layout()
plt.show()

## 9. Key Business Insights <a id='insights'></a>

In [ ]:
# Generate comprehensive insights
print("📊 KEY BUSINESS INSIGHTS")
print("="*80)

# 1. Revenue Insights
print("\n1️⃣ REVENUE PERFORMANCE")
print("-" * 80)
print(f"   • Total Revenue: ${total_revenue:,.2f}")
print(f"   • Total Profit: ${total_profit:,.2f}")
print(f"   • Overall Profit Margin: {profit_margin:.2f}%")
print(f"   • Average Order Value: ${avg_order_value:,.2f}")

# 2. Best Performing Region
best_region = regional_performance.index[0]
best_region_revenue = regional_performance.iloc[0]['Revenue']
print("\n2️⃣ REGIONAL INSIGHTS")
print("-" * 80)
print(f"   • Best Performing Region: {best_region} (${best_region_revenue:,.2f})")
print(f"   • Top Region contributes: {regional_performance.iloc[0]['Revenue %']:.1f}% of total revenue")

# 3. Product Category Insights
best_category = category_performance.index[0]
best_category_revenue = category_performance.iloc[0]['Revenue']
print("\n3️⃣ PRODUCT CATEGORY INSIGHTS")
print("-" * 80)
print(f"   • Top Category: {best_category} (${best_category_revenue:,.2f})")
print(f"   • Most Profitable Category: {category_performance.sort_values('Profit Margin %', ascending=False).index[0]}")
print(f"   • Highest Profit Margin: {category_performance['Profit Margin %'].max():.2f}%")

# 4. Customer Segment Insights
best_segment = segment_performance.index[0]
best_segment_revenue = segment_performance.iloc[0]['Revenue']
print("\n4️⃣ CUSTOMER SEGMENT INSIGHTS")
print("-" * 80)
print(f"   • Top Segment: {best_segment} (${best_segment_revenue:,.2f})")
print(f"   • Highest Avg Order Value: {segment_performance['Avg Order Value'].max():,.2f} ({segment_performance['Avg Order Value'].idxmax()})")

# 5. Seasonal Trends
best_month = monthly_revenue.idxmax()
best_month_revenue = monthly_revenue.max()
print("\n5️⃣ SEASONAL TRENDS")
print("-" * 80)
print(f"   • Best Month: {best_month} (${best_month_revenue:,.2f})")
print(f"   • Peak Quarter: Q{df.groupby('Quarter')['Sales'].sum().idxmax()}")

print("\n" + "="*80)
print("💡 STRATEGIC RECOMMENDATIONS")
print("="*80)
print("   1. Focus marketing efforts on the best-performing regions")
print("   2. Increase inventory for high-revenue product categories")
print("   3. Develop targeted campaigns for high-value customer segments")
print("   4. Leverage seasonal trends for promotional planning")
print("   5. Optimize discount strategies to improve profit margins")
print("="*80)

In [ ]:
# Export summary report
summary_report = pd.DataFrame({
    'Metric': [
        'Total Revenue',
        'Total Profit',
        'Total Orders',
        'Average Order Value',
        'Overall Profit Margin',
        'Best Region',
        'Best Category',
        'Best Segment',
        'Best Month'
    ],
    'Value': [
        f'${total_revenue:,.2f}',
        f'${total_profit:,.2f}',
        f'{total_orders:,}',
        f'${avg_order_value:,.2f}',
        f'{profit_margin:.2f}%',
        best_region,
        best_category,
        best_segment,
        best_month
    ]
})

print("\n📄 Summary Report:")
print(summary_report.to_string(index=False))

# Save summary report
# summary_report.to_csv('../data/summary_report.csv', index=False)
# print("\n✓ Summary report saved to data/summary_report.csv")

---
## Conclusion

This analysis has provided comprehensive insights into retail sales performance across multiple dimensions:

- **Revenue Trends**: Identified monthly and quarterly patterns
- **Regional Performance**: Determined top-performing markets
- **Product Analysis**: Highlighted best-selling categories and products
- **Customer Insights**: Analyzed segment behavior and preferences

These insights form the foundation for data-driven business decisions and strategic planning.

**Next Steps**: Import this data into Power BI for interactive dashboard creation.

---
**Project by:** Aman Panday  
**Contact:** [Your Email/LinkedIn]